# 03 — Variabilidade em escala: curvas de luz do ZTF

Workshop SAB — AI-Scope / ADSS

**Pergunta científica:** como passar de uma seleção espacial para comportamento temporal sem baixar uma base inteira de séries temporais?

`public.ztf_dr22` armazena `hmjd`, `mag`, `magerr` e `catflags` como arrays. O objetivo é selecionar fontes no servidor e transformar apenas as séries retornadas em uma representação adequada para análise temporal.
Associe `upload.targets` a `public.ztf_dr22` dentro de **0,0002°**. Retorne ID/posição da entrada, todas as colunas ZTF (`d.*`) e a separação `sep`. Preserve todas as linhas: cada `filterid` pode estar em uma linha distinta.

`np.asarray` transforma os arrays recebidos em vetores numéricos, e `np.isfinite` remove valores ausentes antes do gráfico. `errorbar` desenha as incertezas `magerr`. Como magnitude menor significa mais brilho, o eixo vertical é invertido.

### ZTF — ZTF DR22
O **ZTF** (Zwicky Transient Facility) é um levantamento temporal realizado com o telescópio Samuel Oschin de 48 polegadas, no Observatório Palomar.

- **Instrumento:** câmera de grande campo com aproximadamente 47 graus quadrados.
- **Dados:** fotometria multiepoch nas bandas `g`, `r` e `i`.
- **ZTF DR22:** publicação de dados fotométricos utilizada para construir curvas de luz.

- **ZTF DR22:** https://irsa.ipac.caltech.edu/data/ZTF/docs/releases/dr22/ztf_release_notes_dr22.pdf

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import adss

BASE_URL = "https://ai-scope.cbpf.br/"

cl = adss.ADSSClient(base_url=BASE_URL)

In [ ]:
targets = pd.read_csv("outputs/cluster_1.csv")

In [ ]:
metadata = cl.get_database_metadata()

table = metadata.get_table("public", "ztf_dr22")
if table is not None:
    print(table.column_names())

In [ ]:
query = """"""  # Complete a consulta AstroQL do enunciado.
if not query:
    raise NotImplementedError("Complete query;")

ztf = cl.query_and_wait(query_text=query, mode="astroql", file=targets, table_name="targets").data
ztf[["id", "ra", "dec", "filterid", "nepochs"]].head()

In [ ]:
ztf

In [ ]:
filter_names = {1: "g", 2: "r", 3: "i"}


# Escolher a estrela com mais filtros e observações
summary = ztf.groupby("input_source_id").agg(
    filters=("filterid", "nunique"),
    epochs=("nepochs", "sum")
)

source_id = summary.sort_values(
    ["filters", "epochs"], ascending=False
).index[0]

source = ztf[ztf.input_source_id == source_id]

# Organizar as curvas de luz por filtro
light_curves = {}

for band, data in source.groupby("filterid"):
    time = np.concatenate(data.hmjd.to_numpy())
    mag = np.concatenate(data.mag.to_numpy())
    err = np.concatenate(data.magerr.to_numpy())

    valid = np.isfinite(time) & np.isfinite(mag) & np.isfinite(err) & (err > 0)

    if valid.any():
        light_curves[band] = (
            time[valid],
            mag[valid],
            err[valid]
        )

# Plotar
fig, ax = plt.subplots(figsize=(9, 4))

names = {1: "g", 2: "r", 3: "i"}

for band, (time, mag, err) in light_curves.items():
    ax.errorbar(
        time, mag, yerr=err,
        fmt=".", ms=3, alpha=0.7,
        label=f"ZTF {names.get(band, band)}"
    )

ax.invert_yaxis()
ax.set(xlabel="HMJD", ylabel="Magnitude", title=f"Gaia source {source_id}")
ax.legend()

plt.show()

### Procurar periodicidade com Lomb–Scargle

`astropy.timeseries.LombScargle` procura sinais periódicos em medidas feitas em **tempos irregulares**. Recebe tempo `t`, magnitude `m` e incerteza `e`; `autopower` calcula a potência em uma grade de frequências. O maior pico fornece um **período candidato** (`1/f`), que pode ser um alias da cadência e precisa de inspeção. Ajustamos uma banda bem amostrada para evitar deslocamentos de magnitude entre filtros e depois dobramos todas as bandas em fase. O período não prova, sozinho, que a fonte é variável periódica.

Referência: [Astropy Lomb–Scargle](https://docs.astropy.org/en/stable/timeseries/lombscargle.html).


In [ ]:
from astropy.timeseries import LombScargle

# Fit the best-sampled filter so offsets between bands do not affect the period.
period_filter = max(light_curves, key=lambda filter_id: len(light_curves[filter_id][0]))
t, m, e = light_curves[period_filter]
frequency, power = LombScargle(t, m, e).autopower(
    minimum_frequency=1/100,
    maximum_frequency=10,
)
best_frequency = frequency[np.argmax(power)]
best_period = 1 / best_frequency
print(f"Candidate period ({filter_names.get(period_filter, period_filter)} band): {best_period:.5f} d")

fig, ax = plt.subplots(figsize=(7, 4))
for filter_id, (t, m, e) in light_curves.items():
    phase = (t % best_period) / best_period
    ax.errorbar(phase, m, yerr=e, fmt=".", ms=3, alpha=0.7,
                label=f"ZTF {filter_names.get(filter_id, filter_id)}")
ax.invert_yaxis()
ax.legend()
ax.set(xlabel="phase", ylabel="magnitude", title=f"Phase-folded Gaia source {source_id}")
plt.show()


### Extensão — Descoberta de objetos variáveis

Podemos caracterizar as curvas de luz do ZTF utilizando propriedades estatísticas (*features*) para identificar objetos com comportamentos incomuns.

**Características extraídas:**
- `amp90`: amplitude entre os percentis 5 e 95, menos sensível a valores extremos.
- `std`: desvio padrão das magnitudes.
- `mad`: desvio absoluto mediano, medida robusta de dispersão.
- `skew`: assimetria da distribuição de magnitudes.
- `range`: diferença entre as magnitudes máxima e mínima.

Cada linha representa uma curva de luz de um registro do catálogo. Consideramos apenas curvas com pelo menos 10 medidas válidas.

**Objetivo:** identificar candidatos com variabilidade incomum, que poderão ser investigados posteriormente com métodos de detecção de anomalias, como *Isolation Forest*.

**Atenção:** amplitudes elevadas também podem ser causadas por erros fotométricos. Os candidatos precisam ser validados.

In [ ]:
from scipy.stats import skew

def variability_features(row):
    mag = np.asarray(row.mag, dtype=float)
    mag = mag[np.isfinite(mag)]

    if len(mag) < 10:
        return pd.Series({
            "amp90": np.nan,
            "std": np.nan,
            "mad": np.nan,
            "skew": np.nan,
            "range": np.nan
        })

    median = np.median(mag)

    return pd.Series({
        "amp90": np.percentile(mag, 95) - np.percentile(mag, 5),
        "std": np.std(mag),
        "mad": np.median(np.abs(mag - median)),
        "skew": skew(mag),
        "range": np.ptp(mag)
    })


# Extrair características das curvas de luz
features = pd.concat([
    ztf[["id", "ra", "dec", "filterid", "nepochs"]],
    ztf.apply(variability_features, axis=1)
], axis=1)

# Objetos com maior amplitude de variabilidade
features.sort_values("amp90", ascending=False).head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    features["std"],
    features["amp90"],
    s=2,
    alpha=0.4
)

ax.set_xlabel("Desvio padrão")
ax.set_ylabel("Amplitude robusta (mag)")
ax.set_title("Variabilidade ZTF")

plt.show()